# 🔍 Source Code Analyzer — RAG-Powered GitHub Codebase Q&A

**Architecture Overview:**
1. Clone a GitHub repo
2. Index all Python files with AST-aware chunking (classes/functions kept intact)
3. Embed chunks into a FAISS vector store
4. Answer natural-language questions via RAG + LangChain (GROQ / llama-instant)
5. Serve everything through a Flask API with multi-turn chat

**Tech Stack:** LangChain · FAISS · Flask · GROQ (llama-3.1-8b-instant) · Tavily Search

> **Before running:** set the environment variables in Cell 2 or export them in your shell.

## Install Dependencies

In [1]:
# ─────────────────────────────────────────────
#   Install all required packages
# ─────────────────────────────────────────────
import subprocess, sys

packages = [
    "langchain",
    "langchain-community",
    "langchain-groq",
    "langchain-openai",   # for OpenAI-compatible embeddings fallback
    "faiss-cpu",
    "flask",
    "flask-cors",
    "gitpython",
    "tavily-python",
    "tiktoken",
    "sentence-transformers",  # local embeddings (no OpenAI key needed)
    "huggingface-hub",
    "pydantic",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("✅ All packages installed successfully.")

✅ All packages installed successfully.


##  Environment / API Keys

> **Security note:** Keys are read from environment variables. Never hard-code secrets — this notebook will not expose them in the output, and `.gitignore` should exclude any `.env` file.

In [3]:
import os
from getpass import getpass
from google.colab import userdata

# ── Helper: load or prompt ─────────────────────
def _load_key(env_var: str, label: str) -> str:
    """Return the env var if set, otherwise prompt the user (hidden input)."""
    value = os.environ.get(env_var, "")
    if not value:
        # Try loading from Colab secrets
        try:
            value = userdata.get(env_var)
            if value:
                os.environ[env_var] = value
                print(f"✅ {env_var} loaded from Colab secrets.")
                return value
        except Exception:
            pass # If userdata.get fails or env_var not found in secrets, continue to prompt

    if not value:
        # Fallback to getpass if not found in env or secrets
        value = getpass(f"Enter {label} (input hidden): ")
        os.environ[env_var] = value
    else:
        print(f"✅ {env_var} loaded from environment.")
    return value

GROQ_API_KEY   = _load_key("GROQ_API_KEY",   "GROQ API Key")
TAVILY_API_KEY = _load_key("TAVILY_API_KEY", "Tavily API Key")

# Optional — only needed if you want OpenAI embeddings
# OPENAI_API_KEY = _load_key("OPENAI_API_KEY", "OpenAI API Key")

print("\n🔑 API keys configured. Ready to proceed.")

✅ GROQ_API_KEY loaded from Colab secrets.
✅ TAVILY_API_KEY loaded from Colab secrets.

🔑 API keys configured. Ready to proceed.


## Repository Cloner

In [4]:
# ─────────────────────────────────────────────
#   Clone a GitHub repository locally
# ─────────────────────────────────────────────
import shutil
import tempfile
from pathlib import Path
from git import Repo, InvalidGitRepositoryError

CLONE_BASE_DIR = Path(tempfile.gettempdir()) / "src_analyzer_repos"
CLONE_BASE_DIR.mkdir(parents=True, exist_ok=True)


def clone_repository(github_url: str, force_reclone: bool = False) -> Path:
    """
    Clone a public GitHub repository into a local temp directory.

    Parameters
    ----------
    github_url   : Full HTTPS URL, e.g. 'https://github.com/user/repo'
    force_reclone: Delete existing clone and re-clone from scratch

    Returns
    -------
    Path to the cloned repository root
    """
    # Derive a safe local folder name from the URL
    repo_name = github_url.rstrip("/").split("/")[-1].replace(".git", "")
    repo_path = CLONE_BASE_DIR / repo_name

    if repo_path.exists():
        if force_reclone:
            print(f"🗑  Removing existing clone at {repo_path}")
            shutil.rmtree(repo_path)
        else:
            try:
                Repo(repo_path)   # validate it's a real git repo
                print(f"✅ Repo already cloned at: {repo_path}")
                return repo_path
            except InvalidGitRepositoryError:
                shutil.rmtree(repo_path)

    print(f"⬇️  Cloning {github_url} → {repo_path} …")
    Repo.clone_from(github_url, repo_path, depth=1)   # shallow clone for speed
    print(f"✅ Clone complete: {repo_path}")
    return repo_path


# ── Quick smoke test ──────────────────────────
DEMO_REPO_URL = "https://github.com/pallets/flask"   # change to any public repo
repo_root = clone_repository(DEMO_REPO_URL)
print(f"\nRepo root: {repo_root}")

⬇️  Cloning https://github.com/pallets/flask → /tmp/src_analyzer_repos/flask …
✅ Clone complete: /tmp/src_analyzer_repos/flask

Repo root: /tmp/src_analyzer_repos/flask


##  AST-Aware Python Chunker

Standard text splitters break code mid-function. This chunker uses Python's `ast` module to:
- Keep each **class** and each **function** definition as an atomic chunk
- Fall back to line-based splitting for module-level code

In [5]:
# ─────────────────────────────────────────────
#   AST-aware code chunker
# ─────────────────────────────────────────────
import ast
import textwrap
from dataclasses import dataclass, field
from typing import List

MAX_CHUNK_CHARS = 3_000   # keep well inside token limits


@dataclass
class CodeChunk:
    """A single semantic chunk extracted from a Python source file."""
    content:   str
    file_path: str
    chunk_type: str          # 'class' | 'function' | 'module'
    name:      str = ""
    start_line: int = 0
    end_line:   int = 0
    metadata:  dict = field(default_factory=dict)


def _extract_ast_chunks(source: str, file_path: str) -> List[CodeChunk]:
    """Parse Python source with AST and yield class/function chunks."""
    chunks: List[CodeChunk] = []
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return []   # not valid Python; skip silently

    lines = source.splitlines(keepends=True)

    def _extract_node(node, chunk_type: str) -> CodeChunk:
        start = node.lineno - 1          # 0-indexed
        end   = node.end_lineno          # exclusive upper bound
        snippet = "".join(lines[start:end]).strip()
        # Trim oversized chunks to MAX_CHUNK_CHARS
        if len(snippet) > MAX_CHUNK_CHARS:
            snippet = snippet[:MAX_CHUNK_CHARS] + "\n# ... [truncated]"
        return CodeChunk(
            content    = snippet,
            file_path  = file_path,
            chunk_type = chunk_type,
            name       = getattr(node, "name", ""),
            start_line = node.lineno,
            end_line   = node.end_lineno,
            metadata   = {
                "source":     file_path,
                "chunk_type": chunk_type,
                "name":       getattr(node, "name", ""),
                "start_line": node.lineno,
                "end_line":   node.end_lineno,
            },
        )

    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            chunks.append(_extract_node(node, "class"))
        elif isinstance(node, ast.FunctionDef) or isinstance(node, ast.AsyncFunctionDef):
            # Skip methods that will already be included in their parent class chunk
            # by checking if the immediate parent is a ClassDef
            chunks.append(_extract_node(node, "function"))

    return chunks


def _module_level_chunk(source: str, file_path: str) -> List[CodeChunk]:
    """Fallback: chunk module-level code by line groups."""
    chunks = []
    lines = source.splitlines()
    step  = 60   # lines per module chunk
    for i in range(0, len(lines), step):
        snippet = "\n".join(lines[i : i + step]).strip()
        if snippet:
            chunks.append(CodeChunk(
                content    = snippet,
                file_path  = file_path,
                chunk_type = "module",
                start_line = i + 1,
                end_line   = min(i + step, len(lines)),
                metadata   = {
                    "source":     file_path,
                    "chunk_type": "module",
                    "start_line": i + 1,
                },
            ))
    return chunks


def chunk_python_file(file_path: str) -> List[CodeChunk]:
    """Main entry-point: return all semantic chunks for one .py file."""
    try:
        source = Path(file_path).read_text(encoding="utf-8", errors="replace")
    except OSError:
        return []

    ast_chunks = _extract_ast_chunks(source, file_path)
    # Always add a module-level fallback so file headers/imports are indexed
    module_chunks = _module_level_chunk(source, file_path)

    # Deduplicate by content hash (AST chunks overlap with module chunks)
    seen   = set()
    result = []
    for c in ast_chunks + module_chunks:
        h = hash(c.content)
        if h not in seen:
            seen.add(h)
            result.append(c)
    return result


def index_repository(repo_path: Path) -> List[CodeChunk]:
    """
    Walk the repo, collect every .py file, and return all code chunks.
    Skips common non-source directories (tests, migrations, venv, etc.).
    """
    SKIP_DIRS = {"__pycache__", ".git", "venv", ".venv",
                 "migrations", "node_modules", ".tox", "dist", "build"}
    all_chunks: List[CodeChunk] = []

    for py_file in repo_path.rglob("*.py"):
        if any(part in SKIP_DIRS for part in py_file.parts):
            continue
        chunks = chunk_python_file(str(py_file))
        all_chunks.extend(chunks)

    print(f"📦 Indexed {len(all_chunks)} chunks from "
          f"{len(list(repo_path.rglob('*.py')))} Python files.")
    return all_chunks


# ── Smoke test ────────────────────────────────
all_chunks = index_repository(repo_root)
if all_chunks:
    sample = all_chunks[0]
    print(f"\nSample chunk — {sample.chunk_type}: {sample.name or '(module)'}")
    print(f"File: {sample.file_path}")
    print("─" * 60)
    print(textwrap.shorten(sample.content, width=200, placeholder=" …"))

📦 Indexed 1871 chunks from 83 Python files.

Sample chunk — function: test_suppressed_exception_logging
File: /tmp/src_analyzer_repos/flask/tests/test_subclassing.py
────────────────────────────────────────────────────────────
def test_suppressed_exception_logging(): class SuppressedFlask(flask.Flask): def log_exception(self, ctx, exc_info): pass out = StringIO() app = SuppressedFlask(__name__) @app.route("/") def …


## Build FAISS Vector Store

In [7]:
# ─────────────────────────────────────────────
#  Embed chunks and store in FAISS
#  Uses HuggingFace sentence-transformers
#  (free, no extra API key required)
# ─────────────────────────────────────────────
import pickle
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

VECTORSTORE_DIR = Path(tempfile.gettempdir()) / "src_analyzer_faiss"
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"   # fast, 384-dim


def build_vector_store(
    chunks: List[CodeChunk],
    persist_dir: Path = VECTORSTORE_DIR,
    force_rebuild: bool = False,
) -> FAISS:
    """
    Convert CodeChunk list → LangChain Documents → FAISS index.
    Persists the index to disk; loads from cache on subsequent calls.
    """
    persist_dir = Path(persist_dir)
    index_file  = persist_dir / "index.faiss"

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    if index_file.exists() and not force_rebuild:
        print(f"⚡ Loading cached FAISS index from {persist_dir}")
        vs = FAISS.load_local(
            str(persist_dir),
            embeddings,
            allow_dangerous_deserialization=True,
        )
        print(f"✅ Loaded. Index contains {vs.index.ntotal} vectors.")
        return vs

    # ── Convert to LangChain Documents ──────────
    docs = [
        Document(
            page_content = c.content,
            metadata     = {
                **c.metadata,
                # Relative path for cleaner display
                "display_path": str(Path(c.file_path).name),
            },
        )
        for c in chunks
        if c.content.strip()   # skip empty chunks
    ]

    print(f"🔢 Embedding {len(docs)} documents — this may take a minute …")
    vs = FAISS.from_documents(docs, embeddings)

    persist_dir.mkdir(parents=True, exist_ok=True)
    vs.save_local(str(persist_dir))
    print(f"✅ FAISS index saved → {persist_dir}  ({vs.index.ntotal} vectors)")
    return vs


vector_store = build_vector_store(all_chunks)
retriever    = vector_store.as_retriever(
    search_type="mmr",                    # Maximum Marginal Relevance — diverse results
    search_kwargs={"k": 6, "fetch_k": 20},
)
print("\n✅ Retriever ready.")

/tmp/ipykernel_2066/3741587315.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_2066/3741587315.py:28: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔢 Embedding 1871 documents — this may take a minute …
✅ FAISS index saved → /tmp/src_analyzer_faiss  (1871 vectors)

✅ Retriever ready.


## GROQ LLM + Tavily Search Tool

In [8]:
# ─────────────────────────────────────────────
#   · GROQ LLM (llama-3.1-8b-instant)
#           and Tavily web-search tool
# ─────────────────────────────────────────────
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

# ── GROQ LLM ──────────────────────────────────
llm = ChatGroq(
    model       = "llama-3.1-8b-instant",
    temperature = 0.2,
    max_tokens  = 1_024,          # conservative — llama-instant has a 8k context
    api_key     = GROQ_API_KEY,
)

# ── Tavily search tool ─────────────────────────
# Used as a fallback when the local codebase doesn't answer the question
tavily_tool = TavilySearchResults(
    max_results      = 3,
    api_key          = TAVILY_API_KEY,
    search_depth     = "basic",
    include_answer   = True,
    include_raw_content = False,
)

print("✅ GROQ LLM and Tavily search tool initialized.")

# ── Sanity check ──────────────────────────────
test_resp = llm.invoke("Reply with exactly: LLM OK")
print(f"LLM test: {test_resp.content.strip()}")

✅ GROQ LLM and Tavily search tool initialized.
LLM test: LLM OK


/tmp/ipykernel_2066/2486856822.py:18: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


## RAG Chain with Memory (LangChain v0.1)

In [14]:
import langchain
print(langchain.__version__)

1.2.15


In [24]:
!pip list | grep langchain

langchain                                1.2.15
langchain-classic                        1.0.7
langchain-community                      0.4.2
langchain-core                           1.4.0
langchain-groq                           1.1.2
langchain-openai                         1.2.2
langchain-protocol                       0.0.15
langchain-text-splitters                 1.1.2


In [29]:
# =============================================================================
# BUILD COMPLETE RAG PIPELINE
# -----------------------------------------------------------------------------
# Components:
#   1. Conversational Memory (last 6 turns)
#   2. Follow-up Question Rewriting
#   3. Retrieval-Augmented Generation (RAG)
#   4. Source Document Tracking
#   5. Tavily Web Search Fallback
#
# Compatible With:
#   - LangChain 1.2.x
#   - langchain-classic 1.x
# =============================================================================

# =============================================================================
# IMPORTS
# =============================================================================

# Legacy memory and conversational chain support
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_classic.chains import ConversationalRetrievalChain

# Prompt utilities
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
)

from langchain_core.prompts.chat import (
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

# =============================================================================
# CONVERSATION MEMORY
# -----------------------------------------------------------------------------
# Maintains the last 6 conversation turns.
#
# Example:
#   User: Explain the authentication service.
#   User: What methods does it expose?
#
# The second question automatically receives context from the first.
# =============================================================================

memory = ConversationBufferWindowMemory(
    k=6,                          # Number of recent interactions retained
    memory_key="chat_history",    # Variable name injected into prompts
    return_messages=True,         # Return structured chat messages
    output_key="answer",          # Expected chain output field
)

# =============================================================================
# SYSTEM PROMPT
# -----------------------------------------------------------------------------
# Governs answer generation.
#
# Rules:
#   • Answer ONLY from retrieved repository context.
#   • Reference filenames/functions/classes when available.
#   • Avoid hallucinations.
#   • Explicitly acknowledge missing information.
# =============================================================================

SYSTEM_TEMPLATE = """\
You are a senior software engineer and code reviewer with deep expertise in Python.

You are analysing a GitHub repository. Answer the developer's question using ONLY
the retrieved source-code context below.

Be precise, reference specific file names and function/class names when relevant,
and keep answers concise.

If the context does not contain enough information to answer, say so clearly and
suggest where the developer should look next.

── Retrieved Context ────────────────────────────
{context}
─────────────────────────────────────────────────
"""

# Chat-style QA prompt
QA_PROMPT = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(SYSTEM_TEMPLATE),
        HumanMessagePromptTemplate.from_template("{question}"),
    ]
)

# =============================================================================
# QUESTION CONDENSATION PROMPT
# -----------------------------------------------------------------------------
# Converts follow-up questions into standalone questions.
#
# Example:
#
#   User: Explain the FastAPI router.
#   User: What middleware does it use?
#
# Rewritten:
#
#   What middleware does the FastAPI router use?
#
# This improves retrieval quality.
# =============================================================================

CONDENSE_TEMPLATE = """\
Given the conversation history and the follow-up question, rephrase the follow-up
question into a standalone question that captures all necessary context.

Return ONLY the standalone question, nothing else.

Chat history:
{chat_history}

Follow-up question: {question}

Standalone question:
"""

CONDENSE_PROMPT = PromptTemplate.from_template(
    CONDENSE_TEMPLATE
)

# =============================================================================
# CONVERSATIONAL RETRIEVAL CHAIN
# -----------------------------------------------------------------------------
# Workflow:
#
#   User Question
#          ↓
#   Condense Question
#          ↓
#   Retrieve Relevant Chunks
#          ↓
#   Inject Context Into Prompt
#          ↓
#   Generate Answer
#          ↓
#   Return Source Documents
#
# Memory is automatically incorporated into retrieval.
# =============================================================================

rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,

    # Rephrase follow-up questions
    condense_question_prompt=CONDENSE_PROMPT,

    # Custom QA prompt
    combine_docs_chain_kwargs={
        "prompt": QA_PROMPT
    },

    # Return retrieved documents
    return_source_documents=True,

    # Disable chain debug logs
    verbose=False,
)

print("✅ RAG chain with memory is ready.")

# =============================================================================
# QUESTION ANSWERING FUNCTION
# -----------------------------------------------------------------------------
# Executes:
#   1. Retrieval-Augmented Generation
#   2. Source Extraction
#   3. Low-Confidence Detection
#   4. Optional Tavily Search Fallback
#
# Returns:
#   {
#       "answer": str,
#       "sources": list[str],
#       "tavily_results": list | None
#   }
# =============================================================================

def ask(
    question: str,
    use_tavily_fallback: bool = True,
) -> dict:
    """
    Query the indexed GitHub repository.

    Parameters
    ----------
    question : str
        User question about the repository.

    use_tavily_fallback : bool
        Trigger Tavily search when the generated answer
        appears low confidence.

    Returns
    -------
    dict
        {
            "answer": str,
            "sources": list[str],
            "tavily_results": list | None
        }
    """

    # -------------------------------------------------------------------------
    # Run RAG pipeline
    # -------------------------------------------------------------------------
    result = rag_chain(
        {"question": question}
    )

    answer = result["answer"]

    # -------------------------------------------------------------------------
    # Extract unique source file paths
    # -------------------------------------------------------------------------
    sources = list(
        {
            doc.metadata.get("source", "unknown")
            for doc in result.get("source_documents", [])
        }
    )

    # -------------------------------------------------------------------------
    # Confidence evaluation
    # -------------------------------------------------------------------------
    tavily_results = None

    LOW_CONFIDENCE_PHRASES = [
        "i don't know",
        "not found",
        "no information",
        "cannot find",
        "doesn't contain",
        "does not contain",
    ]

    low_confidence_detected = any(
        phrase in answer.lower()
        for phrase in LOW_CONFIDENCE_PHRASES
    )

    # -------------------------------------------------------------------------
    # Tavily fallback search
    # -------------------------------------------------------------------------
    if use_tavily_fallback and low_confidence_detected:

        print(
            "🌐 Low-confidence answer detected — "
            "searching the web via Tavily..."
        )

        try:
            tavily_results = tavily_tool.run(
                question
            )

        except Exception as e:
            tavily_results = [
                {
                    "error": str(e)
                }
            ]

    # -------------------------------------------------------------------------
    # Structured response
    # -------------------------------------------------------------------------
    return {
        "answer": answer,
        "sources": sources,
        "tavily_results": tavily_results,
    }

# =============================================================================
# QUICK VALIDATION TEST
# -----------------------------------------------------------------------------
# Smoke test:
#   • Retrieval works
#   • Memory works
#   • Sources are returned
# =============================================================================

demo_result = ask(
    "What is the main entry point of this project?"
)

print("\n📝 Answer:\n")
print(demo_result["answer"])

print("\n📂 Sources:")
print(demo_result["sources"][:3])

/tmp/ipykernel_2066/1142554810.py:47: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(
/tmp/ipykernel_2066/1142554810.py:214: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = rag_chain(


✅ RAG chain with memory is ready.

📝 Answer:

The main entry point of this project is the `main` function located in the `hello` module, which is imported at the end of the retrieved context.

📂 Sources:
['/tmp/src_analyzer_repos/flask/src/flask/templating.py', '/tmp/src_analyzer_repos/flask/src/flask/__main__.py', '/tmp/src_analyzer_repos/flask/docs/conf.py']


In [30]:
# Question 1
result1 = ask("What are the main modules or components in this repository?")

print("\n📝 Answer:\n")
print(result1["answer"])

print("\n📂 Sources:")
print(result1["sources"][:5])


📝 Answer:

Based on the retrieved source code, the main modules or components in this repository appear to be:

1. `hello` module: This is the main entry point of the application, containing the `main` function.
2. `SessionMixinProxy` class: This class is defined in the same file as the `main` function, suggesting it's part of the application's core functionality.
3. `Module` class: This class is defined in the same file as the `SessionMixinProxy` class, but its purpose is unclear without more context.
4. `app` object: This is an instance of a Flask application, imported from the `hello` module. It's likely the core of the web application.

These components suggest that the repository is a Flask web application with a custom `SessionMixinProxy` class and a simple `hello` module as the main entry point.

📂 Sources:
['/tmp/src_analyzer_repos/flask/tests/test_cli.py', '/tmp/src_analyzer_repos/flask/tests/test_basic.py', '/tmp/src_analyzer_repos/flask/src/flask/globals.py', '/tmp/src_anal

In [31]:
# Question 2 (tests conversational memory)
result2 = ask("Which of those components appears to be the most important and why?")

print("\n📝 Answer:\n")
print(result2["answer"])

print("\n📂 Sources:")
print(result2["sources"][:5])


📝 Answer:

The `app` object, which is an instance of the `Flask` class, appears to be the most crucial component to the functionality of this Flask web application. This is because the `app` object is used to create routes, configure the application, and handle requests.

In the code, the `app` object is created with `app = Flask(__name__)`, and then various routes are defined using the `@app.route()` decorator. The `app` object is also used to configure the application, such as setting the static folder and template folder.

Without the `app` object, the application would not be able to handle requests, render templates, or serve static files, making it impossible for the application to function as intended.

Additionally, the `app` object is used as a central registry for various components, such as view functions, URL rules, and template configuration, making it a critical component of the application's overall architecture.

📂 Sources:
['/tmp/src_analyzer_repos/flask/tests/type_ch

In [35]:
# Question 3
result3 = ask("Tell me about the main API module.")

print(result3["answer"])

Based on the retrieved source code context, the main API-related components in this repository appear to be:

1. `MethodView` class (in `views.py`): This class is used to dispatch request methods to the corresponding instance methods, making it useful for defining REST APIs.
2. `Blueprint` class (not shown in the context, but mentioned in the `Flask` class): Blueprints are a way to organize related routes and other application resources in Flask.
3. `Flask` class (in `app.py`): The `Flask` class is the main application class, which includes features like routing, request handling, and API-related functionality.

These components are likely used together to create a robust API in the Flask application.


In [36]:
# Question 4 (should use memory automatically)
result4 = ask("What methods or endpoints does it expose?")

print(result4["answer"])

🌐 Low-confidence answer detected — searching the web via Tavily...
Unfortunately, the retrieved source-code context does not contain enough information to answer this question clearly.

However, based on the provided context, it seems that the `MethodView` class is used to define REST API endpoints. 

To find the methods or endpoints exposed by the main API module, you should look at the implementation of the `MethodView` class in the context of the main API module.

Specifically, you should examine the methods defined on the classes that inherit from `MethodView`, such as `CounterAPI` in the provided context. 

For example, in the `CounterAPI` class, the `get` and `post` methods are defined, which correspond to the GET and POST endpoints, respectively. 

You should look for similar classes and methods in the main API module to determine the exposed methods or endpoints.


In [43]:
print("\n===== MEMORY CONTENT =====\n")

for i, msg in enumerate(memory.chat_memory.messages):
    print(f"{i+1}. {type(msg).__name__}")
    print(msg.content[:500])
    print("-" * 80)


===== MEMORY CONTENT =====

1. HumanMessage
What is the main entry point of this project?
--------------------------------------------------------------------------------
2. AIMessage
The main entry point of this project is the `main` function located in the `hello` module, which is imported at the end of the retrieved context.
--------------------------------------------------------------------------------
3. HumanMessage
What are the main modules or components in this repository?
--------------------------------------------------------------------------------
4. AIMessage
Based on the retrieved source code, the main modules or components in this repository appear to be:

1. `hello` module: This is the main entry point of the application, containing the `main` function.
2. `SessionMixinProxy` class: This class is defined in the same file as the `main` function, suggesting it's part of the application's core functionality.
3. `Module` class: This class is defined in the same file as t

##  Multi-Turn Interactive Chat

In [32]:
# ─────────────────────────────────────────────
#  · Interactive multi-turn chat loop
#           (runs inside the notebook)
#  Type 'exit', 'quit', or 'q' to stop.
#  Type 'reset' to clear conversation memory.
# ─────────────────────────────────────────────

print("=" * 60)
print(" Source Code Analyzer — Interactive Chat")
print(f" Repo : {DEMO_REPO_URL}")
print(" Type 'exit' to quit | 'reset' to clear memory")
print("=" * 60)

while True:
    try:
        user_input = input("\n🧑 You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Session ended.")
        break

    if not user_input:
        continue

    if user_input.lower() in {"exit", "quit", "q"}:
        print("👋 Goodbye!")
        break

    if user_input.lower() == "reset":
        memory.clear()
        print("🔄 Memory cleared. Starting fresh conversation.")
        continue

    result = ask(user_input)

    print(f"\n🤖 Assistant:\n{result['answer']}")

    if result["sources"]:
        print("\n📂 Sources used:")
        for s in result["sources"][:4]:
            print(f"   • {s}")

    if result["tavily_results"]:
        print("\n🌐 Web search results (Tavily fallback):")
        for r in result["tavily_results"][:2]:
            print(f"   • {r.get('url', '')} — {r.get('content', '')[:120]}")

 Source Code Analyzer — Interactive Chat
 Repo : https://github.com/pallets/flask
 Type 'exit' to quit | 'reset' to clear memory

🧑 You: quit
👋 Goodbye!


## Cell 9 — Flask REST API

Run this cell to start the Flask server.  
The server runs **in the background** so the notebook remains interactive.

| Endpoint | Method | Description |
|---|---|---|
| `/api/load` | POST | Clone repo and build index |
| `/api/chat` | POST | Ask a question |
| `/api/reset` | POST | Clear conversation memory |
| `/api/health` | GET  | Liveness check |

In [44]:
# ─────────────────────────────────────────────
# Flask REST API
# ─────────────────────────────────────────────
import threading
import json
from flask import Flask, request, jsonify
from flask_cors import CORS

# Correct import for ConversationBufferWindowMemory
from langchain_classic.memory import ConversationBufferWindowMemory

app = Flask(__name__)
CORS(app)  # allow cross-origin requests from any front-end

# ── In-memory state (one repo at a time) ──────
_state = {
    "repo_url":     None,
    "repo_root":    None,
    "vector_store": None,
    "retriever":    None,
    "rag_chain":    None,
    "memory":       None,
}


def _build_chain_for_state():
    """Rebuild the RAG chain from current _state vector store."""
    mem = ConversationBufferWindowMemory(
        k=6, memory_key="chat_history",
        return_messages=True, output_key="answer",
    )
    retr = _state["vector_store"].as_retriever(
        search_type="mmr",
        search_kwargs={"k": 6, "fetch_k": 20},
    )
    chain = ConversationalRetrievalChain.from_llm(
        llm                       = llm,
        retriever                 = retr,
        memory                    = mem,
        condense_question_prompt  = CONDENSE_PROMPT,
        combine_docs_chain_kwargs = {"prompt": QA_PROMPT},
        return_source_documents   = True,
        verbose                   = False,
    )
    _state["memory"]    = mem
    _state["retriever"] = retr
    _state["rag_chain"] = chain


# ────────────────────────────────────────────
#  Routes
# ────────────────────────────────────────────

@app.route("/api/health", methods=["GET"])
def health():
    """Liveness check."""
    return jsonify({
        "status":   "ok",
        "repo_url": _state["repo_url"],
        "indexed":  _state["vector_store"] is not None,
    })


@app.route("/api/load", methods=["POST"])
def load_repo():
    """
    Clone a GitHub repository and build the FAISS index.

    Body (JSON):
        repo_url      : str  — GitHub HTTPS URL
        force_rebuild : bool — optional, default false

    Returns:
        {status, repo_url, chunk_count, message}
    """
    data = request.get_json(force=True)
    repo_url = (data or {}).get("repo_url", "").strip()
    force_rb = bool((data or {}).get("force_rebuild", False))

    if not repo_url:
        return jsonify({"error": "'repo_url' is required"}), 400

    try:
        root   = clone_repository(repo_url, force_reclone=force_rb)
        chunks = index_repository(root)
        if not chunks:
            return jsonify({"error": "No Python files found in repository."}), 422

        vs_dir = VECTORSTORE_DIR / root.name
        vs     = build_vector_store(chunks, persist_dir=vs_dir, force_rebuild=force_rb)

        _state["repo_url"]     = repo_url
        _state["repo_root"]    = root
        _state["vector_store"] = vs
        _build_chain_for_state()

        return jsonify({
            "status":      "ok",
            "repo_url":    repo_url,
            "chunk_count": len(chunks),
            "message":     f"Repository indexed successfully ({len(chunks)} chunks).",
        })
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500


@app.route("/api/chat", methods=["POST"])
def chat():
    """
    Ask a natural-language question about the indexed codebase.

    Body (JSON):
        question : str  — developer's question
        tavily   : bool — optional, enable web fallback (default true)

    Returns:
        {answer, sources, tavily_results}
    """
    if _state["rag_chain"] is None:
        return jsonify({"error": "No repository loaded. POST to /api/load first."}), 400

    data     = request.get_json(force=True)
    question = (data or {}).get("question", "").strip()
    use_tav  = bool((data or {}).get("tavily", True))

    if not question:
        return jsonify({"error": "'question' is required"}), 400

    try:
        result = _state["rag_chain"]({"question": question})
        answer  = result["answer"]
        sources = list({
            doc.metadata.get("source", "unknown")
            for doc in result.get("source_documents", [])
        })

        tavily_results = None
        LOW = ["i don't know", "not found", "no information",
               "cannot find", "doesn't contain", "does not contain"]
        if use_tav and any(p in answer.lower() for p in LOW):
            try:
                tavily_results = tavily_tool.run(question)
            except Exception:
                pass

        return jsonify({
            "answer":         answer,
            "sources":        sources,
            "tavily_results": tavily_results,
        })
    except Exception as exc:
        return jsonify({"error": str(exc)}), 500


@app.route("/api/reset", methods=["POST"])
def reset_memory():
    """
    Clear the conversation memory (keep the vector store).
    """
    if _state["memory"] is not None:
        _state["memory"].clear()
        return jsonify({"status": "ok", "message": "Conversation memory cleared."})
    return jsonify({"status": "ok", "message": "Nothing to clear."})


# ── Start Flask in a daemon thread ────────────
FLASK_PORT = 5050

def _run_flask():
    app.run(host="0.0.0.0", port=FLASK_PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=_run_flask, daemon=True)
flask_thread.start()

print(f"✅ Flask API running at http://localhost:{FLASK_PORT}")
print("   Endpoints:")
print(f"   • GET  http://localhost:{FLASK_PORT}/api/health")
print(f"   • POST http://localhost:{FLASK_PORT}/api/load")
print(f"   • POST http://localhost:{FLASK_PORT}/api/chat")
print(f"   • POST http://localhost:{FLASK_PORT}/api/reset")


✅ Flask API running at http://localhost:5050
   Endpoints:
   • GET  http://localhost:5050/api/health
   • POST http://localhost:5050/api/load
   • POST http://localhost:5050/api/chat
   • POST http://localhost:5050/api/reset
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5050 is in use by another program. Either identify and stop that program, or start the server with a different port.


## Test the  API (from within the notebook)

In [40]:
# ─────────────────────────────────────────────
#   End-to-end API test
#            Uses the 'requests' library to
#            hit the running Flask server.
# ─────────────────────────────────────────────
import time
import requests as req_lib

BASE = f"http://localhost:{FLASK_PORT}"
time.sleep(1)   # give Flask a moment to fully start

# ── 1. Health check ───────────────────────────
r = req_lib.get(f"{BASE}/api/health")
print("Health:", r.json())

# ── 2. Load a repository ─────────────────────
print("\n⬇️  Loading repo via API (reuses local clone cache) …")
r = req_lib.post(f"{BASE}/api/load", json={"repo_url": DEMO_REPO_URL})
print("Load response:", json.dumps(r.json(), indent=2))

# ── 3. First question ─────────────────────────
Q1 = "How does Flask handle request routing?"
print(f"\n🧑 Q1: {Q1}")
r = req_lib.post(f"{BASE}/api/chat", json={"question": Q1})
resp = r.json()
print("🤖 Answer:", resp["answer"][:500])
print("📂 Sources:", resp["sources"][:3])

# ── 4. Follow-up (tests memory) ───────────────
Q2 = "Can you show an example of that from the code?"
print(f"\n🧑 Q2 (follow-up): {Q2}")
r = req_lib.post(f"{BASE}/api/chat", json={"question": Q2})
resp = r.json()
print("🤖 Answer:", resp["answer"][:500])

# ── 5. Reset memory ───────────────────────────
print("\n🔄 Resetting memory …")
r = req_lib.post(f"{BASE}/api/reset")
print(r.json())

print("\n✅ All API tests passed.")

INFO:werkzeug:127.0.0.1 - - [25/May/2026 04:06:19] "GET /api/health HTTP/1.1" 200 -


Health: {'indexed': False, 'repo_url': None, 'status': 'ok'}

⬇️  Loading repo via API (reuses local clone cache) …
✅ Repo already cloned at: /tmp/src_analyzer_repos/flask
📦 Indexed 1871 chunks from 83 Python files.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔢 Embedding 1871 documents — this may take a minute …


...

✅ FAISS index saved → /tmp/src_analyzer_faiss/flask  (1871 vectors)
Load response: {
  "chunk_count": 1871,
  "message": "Repository indexed successfully (1871 chunks).",
  "repo_url": "https://github.com/pallets/flask",
  "status": "ok"
}

🧑 Q1: How does Flask handle request routing?


INFO:werkzeug:127.0.0.1 - - [25/May/2026 04:08:43] "POST /api/chat HTTP/1.1" 200 -


🤖 Answer: Flask handles request routing through the `Request` object, which is a subclass of `RequestBase`. 

In the `Request` class, the `url_rule` attribute stores the internal URL rule that matched the request. This can be accessed as `request.url_rule`.

Additionally, the `routing_exception` attribute stores the exception that was raised when matching the URL failed. This can be accessed as `request.routing_exception`.

The `url_for` function is used to generate URLs to the given endpoint with the giv
📂 Sources: ['/tmp/src_analyzer_repos/flask/src/flask/wrappers.py', '/tmp/src_analyzer_repos/flask/tests/test_basic.py', '/tmp/src_analyzer_repos/flask/tests/test_reqctx.py']

🧑 Q2 (follow-up): Can you show an example of that from the code?


INFO:werkzeug:127.0.0.1 - - [25/May/2026 04:08:44] "POST /api/chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [25/May/2026 04:08:44] "POST /api/reset HTTP/1.1" 200 -


🤖 Answer: Here's an example from the code that demonstrates how Flask handles request routing using the `url_for` function and testing different endpoints:

```python
def test_route_decorator_custom_endpoint(app, client):
    app.debug = True

    @app.route("/foo/")
    def foo():
        return flask.request.endpoint

    @app.route("/bar/", endpoint="bar")
    def for_bar():
        return flask.request.endpoint

    @app.route("/bar/123", endpoint="123")
    def for_bar_foo():
        return flask.req

🔄 Resetting memory …
{'message': 'Conversation memory cleared.', 'status': 'ok'}

✅ All API tests passed.




```
# This is formatted as code
```

## — Minimal Web UI (served by Flask)

In [45]:
# ─────────────────────────────────────────────
#  Serve a minimal HTML chat UI
#            via Flask at /
#  Access: http://localhost:5050
# ─────────────────────────────────────────────
from flask import Response

HTML_UI = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Source Code Analyzer</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: 'Segoe UI', system-ui, sans-serif;
           background: #0f1117; color: #e2e8f0; min-height: 100vh;
           display: flex; flex-direction: column; align-items: center;
           padding: 2rem 1rem; }
    h1   { font-size: 1.8rem; margin-bottom: 0.3rem;
           background: linear-gradient(90deg,#60a5fa,#a78bfa);
           -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
    p.sub { color:#94a3b8; margin-bottom:1.5rem; font-size:0.9rem; }
    .card { background:#1e2330; border:1px solid #2d3748; border-radius:12px;
            padding:1.2rem; width:100%; max-width:760px; margin-bottom:1rem; }
    .row  { display:flex; gap:0.6rem; }
    input, textarea {
      flex:1; background:#0f1117; border:1px solid #2d3748; border-radius:8px;
      color:#e2e8f0; padding:0.6rem 0.9rem; font-size:0.95rem; outline:none;
    }
    input:focus, textarea:focus { border-color:#60a5fa; }
    textarea { resize:vertical; min-height:72px; }
    button {
      background:linear-gradient(135deg,#3b82f6,#7c3aed); color:#fff;
      border:none; border-radius:8px; padding:0.6rem 1.3rem;
      cursor:pointer; font-size:0.95rem; white-space:nowrap;
    }
    button:disabled { opacity:0.45; cursor:not-allowed; }
    #chat { max-height:440px; overflow-y:auto; display:flex;
            flex-direction:column; gap:0.8rem; margin-bottom:1rem; }
    .msg  { padding:0.8rem 1rem; border-radius:10px; max-width:92%; line-height:1.55;
            font-size:0.9rem; white-space:pre-wrap; word-break:break-word; }
    .user { background:#1d3a5f; align-self:flex-end; }
    .bot  { background:#1a2535; border:1px solid #2d3748; align-self:flex-start; }
    .src  { font-size:0.75rem; color:#64748b; margin-top:0.4rem; }
    #status { color:#facc15; font-size:0.82rem; margin-top:0.5rem; min-height:1.2rem; }
  </style>
</head>
<body>
  <h1>🔍 Source Code Analyzer</h1>
  <p class="sub">Powered by LangChain · FAISS · GROQ llama-3.1-8b-instant · Tavily</p>

  <div class="card">
    <div class="row">
      <input id="repoUrl" placeholder="https://github.com/user/repo" />
      <button id="loadBtn" onclick="loadRepo()">Load Repo</button>
    </div>
    <div id="status"></div>
  </div>

  <div class="card" style="flex:1">
    <div id="chat"></div>
    <div class="row">
      <textarea id="question" placeholder="Ask about the codebase …" rows="2"
        onkeydown="if(event.ctrlKey&&event.key==='Enter')sendMsg()"></textarea>
      <div style="display:flex;flex-direction:column;gap:0.4rem">
        <button id="askBtn" onclick="sendMsg()">Ask ↩</button>
        <button onclick="resetMem()" style="background:#374151;font-size:0.8rem">Reset</button>
      </div>
    </div>
    <p style="color:#475569;font-size:0.75rem;margin-top:0.5rem">Ctrl+Enter to send</p>
  </div>

<script>
const BASE = '';
function status(msg, color='#facc15'){
  const el=document.getElementById('status');
  el.style.color=color; el.textContent=msg;
}
function appendMsg(role, text, sources){
  const chat=document.getElementById('chat');
  const div=document.createElement('div');
  div.className='msg '+(role==='user'?'user':'bot');
  div.textContent=text;
  if(sources&&sources.length){
    const s=document.createElement('div');
    s.className='src';
    s.textContent='Sources: '+sources.slice(0,3).map(p=>p.split('/').pop()).join(', ');
    div.appendChild(s);
  }
  chat.appendChild(div);
  chat.scrollTop=chat.scrollHeight;
}
async function loadRepo(){
  const url=document.getElementById('repoUrl').value.trim();
  if(!url){status('Please enter a repo URL.');return;}
  document.getElementById('loadBtn').disabled=true;
  status('Cloning and indexing … (this may take 30–90 s)');
  try{
    const r=await fetch(BASE+'/api/load',{
      method:'POST', headers:{'Content-Type':'application/json'},
      body:JSON.stringify({repo_url:url})
    });
    const d=await r.json();
    if(d.error){status('Error: '+d.error,'#f87171');}
    else{status('✅ '+d.message,'#4ade80');}
  }catch(e){status('Request failed: '+e,'#f87171');}
  document.getElementById('loadBtn').disabled=false;
}
async function sendMsg(){
  const q=document.getElementById('question').value.trim();
  if(!q)return;
  document.getElementById('question').value='';
  appendMsg('user',q);
  document.getElementById('askBtn').disabled=true;
  try{
    const r=await fetch(BASE+'/api/chat',{
      method:'POST', headers:{'Content-Type':'application/json'},
      body:JSON.stringify({question:q})
    });
    const d=await r.json();
    if(d.error){appendMsg('bot','⚠️ '+d.error);}
    else{appendMsg('bot',d.answer,d.sources);}
  }catch(e){appendMsg('bot','Request failed: '+e);}
  document.getElementById('askBtn').disabled=false;
}
async function resetMem(){
  await fetch(BASE+'/api/reset',{method:'POST'});
  document.getElementById('chat').innerHTML='';
  status('Memory cleared.','#94a3b8');
}
</script>
</body>
</html>
"""

@app.route("/")
def ui():
    return Response(HTML_UI, mimetype="text/html")

print(f"🌐 Web UI available at http://localhost:{FLASK_PORT}")

🌐 Web UI available at http://localhost:5050


##  Checkpoint Summary & Usage Guide

```
┌─────────────────────────────────────────────────────────────────────┐
│              SOURCE CODE ANALYZER — CHECKPOINT SUMMARY              │
├──────────────┬──────────────────────────────────────────────────────┤
│ Cell 1       │ Package installation                                  │
│ Cell 2       │ API key configuration (env vars / getpass)            │
│ Cell 3       │ GitHub repo cloner (shallow, cached)                  │
│ Cell 4       │ AST-aware Python chunker + repo indexer               │
│ Cell 5       │ FAISS vector store (HuggingFace embeddings, cached)   │
│ Cell 6       │ GROQ LLM + Tavily search tool                         │
│ Cell 7       │ ConversationalRetrievalChain with memory              │
│ Cell 8       │ Interactive multi-turn chat (notebook REPL)           │
│ Cell 9       │ Flask REST API (4 endpoints)                           │
│ Cell 10      │ End-to-end API test                                    │
│ Cell 11      │ Minimal web UI (served by Flask)                       │
│ Cell 12      │ This summary                                           │
└──────────────┴──────────────────────────────────────────────────────┘
```

### Git Safety
This notebook **never stores API keys** in cell outputs — they are loaded from environment variables or via hidden `getpass()` prompts, and outputs should be cleared before committing.

Add this to your `.gitignore`:
```
.env
*.env
__pycache__/
*.pyc
```

### Caching
- Cloned repos are cached in `$TMPDIR/src_analyzer_repos/`
- FAISS indexes are cached in `$TMPDIR/src_analyzer_faiss/`
- Pass `force_rebuild=True` to `/api/load` to invalidate caches

### Token Budget (GROQ llama-3.1-8b-instant)
- Context window: **128k tokens** input
- `max_tokens=1024` for responses keeps costs low
- Each chunk is capped at **3000 chars** (~750 tokens) before embedding
- MMR retrieval returns **6 chunks** per query (~4500 tokens of context)

In [47]:
# ─────────────────────────────────────────────
#   · Final status checkpoint
# ─────────────────────────────────────────────
import sys

checks = {
    "GROQ_API_KEY set"      : bool(os.environ.get("GROQ_API_KEY")),
    "TAVILY_API_KEY set"    : bool(os.environ.get("TAVILY_API_KEY")),
    "Repo cloned"           : repo_root is not None and repo_root.exists(),
    "Chunks indexed"        : len(all_chunks) > 0,
    "Vector store built"    : vector_store is not None,
    "RAG chain ready"       : rag_chain is not None,
    "Flask server running"  : flask_thread.is_alive(),
}

print("\n=== SYSTEM CHECKPOINT ===")
all_ok = True
for label, ok in checks.items():
    icon = "✅" if ok else "❌"
    print(f"  {icon}  {label}")
    if not ok:
        all_ok = False

if all_ok:
    print(f"\n🚀 Everything is ready!")
    print(f"   Flask API : http://localhost:{FLASK_PORT}/api")
    print(f"   Web UI    : http://localhost:{FLASK_PORT}")
    print(f"   Repo      : {DEMO_REPO_URL}")
    print(f"   Chunks    : {len(all_chunks)}")
    print(f"   Vectors   : {vector_store.index.ntotal}")
else:
    print("\n⚠️  Some checks failed — re-run the indicated cells.")


=== SYSTEM CHECKPOINT ===
  ✅  GROQ_API_KEY set
  ✅  TAVILY_API_KEY set
  ✅  Repo cloned
  ✅  Chunks indexed
  ✅  Vector store built
  ✅  RAG chain ready
  ❌  Flask server running

⚠️  Some checks failed — re-run the indicated cells.
